In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from umap import UMAP

In [ ]:
df_HHZ_2022 = pd.read_parquet("year_TERZ_parquets/TERZ_HHZ_hw12_ont25_2022.parquet")
df_HHZ_2023 = pd.read_parquet("year_TERZ_parquets/TERZ_HHZ_hw12_ont25_2023.parquet")
df_HHZ_2024 = pd.read_parquet("year_TERZ_parquets/TERZ_HHZ_hw12_ont25_2024.parquet")
df_HHZ_2025 = pd.read_parquet("year_TERZ_parquets/TERZ_HHZ_hw12_ont25_2025.parquet")

df_HHZ = pd.concat([df_HHZ_2022, df_HHZ_2023, df_HHZ_2024, df_HHZ_2025], ignore_index=True)
print(f"Combined dataset shape: {df_HHZ.shape}")

In [ ]:
df_HHE_2022 = pd.read_parquet("year_TERZ_parquets/TERZ_HHE_hw12_ont25_2022.parquet")
df_HHE_2023 = pd.read_parquet("year_TERZ_parquets/TERZ_HHE_hw12_ont25_2023.parquet")
df_HHE_2024 = pd.read_parquet("year_TERZ_parquets/TERZ_HHE_hw12_ont25_2024.parquet")
df_HHE_2025 = pd.read_parquet("year_TERZ_parquets/TERZ_HHE_hw12_ont25_2025.parquet")

df_HHE = pd.concat([df_HHE_2022, df_HHE_2023, df_HHE_2024, df_HHE_2025], ignore_index=True)
print(f"Combined dataset shape: {df_HHE.shape}")

In [ ]:
df_HHN_2022 = pd.read_parquet("year_TERZ_parquets/TERZ_HHN_hw12_ont25_2022.parquet")
df_HHN_2023 = pd.read_parquet("year_TERZ_parquets/TERZ_HHN_hw12_ont25_2023.parquet")
df_HHN_2024 = pd.read_parquet("year_TERZ_parquets/TERZ_HHN_hw12_ont25_2024.parquet")
df_HHN_2025 = pd.read_parquet("year_TERZ_parquets/TERZ_HHN_hw12_ont25_2025.parquet")

df_HHN = pd.concat([df_HHN_2022, df_HHN_2023, df_HHN_2024, df_HHN_2025], ignore_index=True)
print(f"Combined dataset shape: {df_HHN.shape}")

In [ ]:
HHZ_PATH = Path("TERZ_HHZ_ont25/TERZ_spectrograms_matrix_results_cut_hw12_ont25")
HHN_PATH = Path("TERZ_HHN_ont25/TERZ_HHN_spectrograms_matrix_results_cut_hw12_ont25")
HHE_PATH = Path("TERZ_HHE_ont25/TERZ_spectrograms_matrix_results_cut_hw12_ont25")

In [ ]:
df = pd.concat([df_HHZ, df_HHE, df_HHN], ignore_index=True)
print(f"All directions combined: {df.shape}")

print("\nTriggers per direction and year")
print(pd.crosstab(df["channel"], df["year"], margins=True, margins_name="total").to_string())

In [ ]:
feature_columns = [col for col in df.columns if col.startswith("feature_")]
if len(feature_columns) == 0:
    raise ValueError("No feature_* columns found.")

# the three directions must have the same feature_* columns, otherwise the concat above fills the gaps with NaN
for name, d in (("HHZ", df_HHZ), ("HHE", df_HHE), ("HHN", df_HHN)):
    if set(col for col in d.columns if col.startswith("feature_")) != set(feature_columns):
        raise ValueError(f"{name} has different feature_* columns than the other directions.")

X = df[feature_columns].to_numpy(dtype=np.float64)
if np.isnan(X).any():
    raise ValueError("NaN found in the feature matrix.")

print(f"Feature matrix: X.shape = {X.shape}")

In [ ]:
X_max = X.max(axis=1, keepdims=True)
X_norm = X / X_max
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_norm)
print(f"Scaled matrix: {X_scaled.shape}") 

In [ ]:
RANDOM_STATE = 42
METRIC = "euclidean"
PERPLEXITY = 400   #first attempt: sqrt(N), N = triggers of the three directions
EARLY_EXAGGERATION = 14
print(f"N = {X_scaled.shape[0]} -> PERPLEXITY = {PERPLEXITY}")

In [ ]:
CACHE_DIR = Path("TERZ_12s_all_directions_embeddings_cache")      
CACHE_DIR.mkdir(exist_ok=True)

In [ ]:
FORCE_RECOMPUTE = False   # True = ignore the cache and recompute (the cache file is overwritten)

def load_or_compute(cache_path, compute_fn, n_rows, force=False):
    """Load a cached (n_rows, 2) embedding if there is one, otherwise compute it and save it."""
    if cache_path.exists() and not force:
        emb = np.load(cache_path)
        if emb.shape == (n_rows, 2):
            print(f"[cache] loaded {cache_path.name}")
            return emb
        print(f"[cache] {cache_path.name} has shape {emb.shape}, expected ({n_rows}, 2) -> recomputing")
    print(f"[cache] no valid cache for {cache_path.name} -> computing")
    emb = compute_fn()
    tmp = cache_path.with_suffix(".tmp.npy")   # write to a temp file first, so an interrupted run leaves no broken cache
    np.save(tmp, emb)
    tmp.replace(cache_path)
    print(f"[cache] saved {cache_path.name}")
    return emb

In [ ]:
tag = f"tsne_perp{PERPLEXITY}_ee{EARLY_EXAGGERATION}_{METRIC}_rs{RANDOM_STATE}"
tsne_cache = CACHE_DIR / f"TERZ_hw12_ont25_all_directions_{tag}.npy"


def run_tsne():
    tsne = TSNE(n_components=2, perplexity=PERPLEXITY, early_exaggeration=EARLY_EXAGGERATION,
                init="pca", learning_rate="auto", random_state=RANDOM_STATE, metric=METRIC)
    return tsne.fit_transform(X_scaled)


X_tsne = load_or_compute(tsne_cache, run_tsne, n_rows=len(X_scaled), force=FORCE_RECOMPUTE)

In [ ]:
df_tSNE = pd.DataFrame(X_tsne, columns=['tSNE_1', 'tSNE_2'])
df_tSNE.insert(loc=0, column='year', value=df['year'])
df_tSNE.insert(loc=1, column='station', value=df['station'])
df_tSNE.insert(loc=2, column='channel', value=df['channel'])
df_tSNE.insert(loc=3, column='file', value=df['file'])
df_tSNE.insert(loc=4, column='date', value=df['date'])
df_tSNE.insert(loc=5, column='month_year', value=df['month_year'])
df_tSNE.insert(loc=6, column='trigger_time', value=df['trigger_time'])
df_tSNE.insert(loc=7, column='q_value', value=df['q_value'])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],s=5,alpha=0.5)
ax.set_title(f"tSNE | Perplexity = {PERPLEXITY} | Early Exaggeration = {EARLY_EXAGGERATION} | PCA Components = No PCA | Metric = {METRIC}")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")

plt.tight_layout()
plt.show()

In [ ]:
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

In [ ]:
tag = f"umap_nn{UMAP_N_NEIGHBORS}_md{UMAP_MIN_DIST}_{METRIC}_rs{RANDOM_STATE}"
umap_cache = CACHE_DIR / f"TERZ_hw12_ont25_all_directions_{tag}.npy"


def run_umap():
    reducer = UMAP(n_components=2, n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                   metric=METRIC, random_state=RANDOM_STATE)
    return reducer.fit_transform(X_scaled)


X_umap = load_or_compute(umap_cache, run_umap, n_rows=len(X_scaled), force=FORCE_RECOMPUTE)

In [ ]:
df_umap = pd.DataFrame(X_umap, columns=['umap_1', 'umap_2'])
df_umap.insert(loc=0, column='year', value=df['year'])
df_umap.insert(loc=1, column='station', value=df['station'])
df_umap.insert(loc=2, column='channel', value=df['channel'])
df_umap.insert(loc=3, column='file', value=df['file'])
df_umap.insert(loc=4, column='date', value=df['date'])
df_umap.insert(loc=5, column='month_year', value=df['month_year'])
df_umap.insert(loc=6, column='trigger_time', value=df['trigger_time'])
df_umap.insert(loc=7, column='q_value', value=df['q_value'])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1],s=5,alpha=0.5)
ax.set_title(f"uamp | N_Neighbors = {UMAP_N_NEIGHBORS} | Min Dist. = {UMAP_MIN_DIST} | PCA Components = No PCA | Metric = {METRIC}")
ax.set_xlabel("umap 1")
ax.set_ylabel("umap 2")

plt.tight_layout()
plt.show()

In [ ]:
EMBEDDING_FOR_CLASSES = "tsne"     # "tsne" or "umap"
HDBSCAN_MIN_CLUSTER_SIZE = 750
HDBSCAN_MIN_SAMPLES = 200

In [ ]:
from sklearn.cluster import HDBSCAN

In [ ]:
if EMBEDDING_FOR_CLASSES == "tsne":
    E, df_emb, emb_name = X_tsne, df_tSNE, "t-SNE"
elif EMBEDDING_FOR_CLASSES == "umap":
    E, df_emb, emb_name = X_umap, df_umap, "UMAP"
else:
    raise ValueError("EMBEDDING_FOR_CLASSES must be 'tsne' or 'umap'")

hdb = HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, min_samples=HDBSCAN_MIN_SAMPLES)
labels = hdb.fit_predict(E)              # -1 = noise
df_emb["hdbscan_class"] = labels

n_classes = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int((labels == -1).sum())
print(f"HDBSCAN on {emb_name}: {n_classes} classes, noise = {n_noise} / {len(labels)} ({100 * n_noise / len(labels):.1f}%)")

In [ ]:
def plot_clusters(ax, E, labels, title, xlabel, ylabel):
    noise = labels == -1
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    ax.scatter(E[noise, 0], E[noise, 1], c="lightgray", s=8, alpha=0.5, label="noise")
    ax.scatter(E[~noise, 0], E[~noise, 1], c=labels[~noise], cmap="viridis", s=8, alpha=0.7)
    ax.set_title(f"{title}\n{n_clusters} clusters, {noise.sum()} noise points")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if noise.any():
        ax.legend(loc="best", markerscale=2)
    return n_clusters, int(noise.sum())

In [ ]:
fig = plt.figure(figsize=(18, 16))

ax = fig.add_subplot(2, 2, 1)
ax.scatter(X_tsne[:, 0], X_tsne[:, 1], s=5, alpha=0.5)
n_hdb, noise_hdb = plot_clusters(ax, E, labels, "HDBSCAN", "tSNE-1", "tSNE-2")
fig.suptitle(f"All years, all directions | ont=25 | tSNE | {E.shape[0]} triggers", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.97])

plt.show()

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.lines import Line2D
from PIL import Image

In [ ]:
# direction (value of the "channel" column) -> folder that contains its qtransform PDFs (the paths are defined at the top of the notebook)
RESULTS_ROOTS = {"HHZ": HHZ_PATH, "HHE": HHE_PATH, "HHN": HHN_PATH}
_unknown = set(df["channel"].unique()) - set(RESULTS_ROOTS)
if _unknown:
    raise ValueError(f"No qtransform folder for channel(s) {sorted(_unknown)}: add them to RESULTS_ROOTS.")
PDF_INDEX_ORDER = "df"
N_THUMBS_PER_CLASS = 30
MIN_DIST_FRAC = 0.04       # minimum distance between two thumbnails, as a fraction of the plot span
THUMB_PX = 120             # width in pixels of each rendered thumbnail
THUMB_ZOOM = 0.35          # size of the thumbnails on the figure
THUMB_SEED = 0

In [ ]:
DAY_KEY = ["station", "channel", "date"]
if PDF_INDEX_ORDER == "df":
    df_emb["pdf_index"] = df_emb.groupby(DAY_KEY, sort=False).cumcount()
elif PDF_INDEX_ORDER == "time":
    _t = pd.to_datetime(df_emb["trigger_time"], utc=True)
    df_emb["pdf_index"] = df_emb.assign(_t=_t).groupby(DAY_KEY, sort=False)["_t"].rank(method="first").astype(int) - 1
else:
    raise ValueError("PDF_INDEX_ORDER must be 'df' or 'time'")


def qtransform_pdf_path(row):
    """Path of the qtransform PDF of one row of df_emb.
    Layout: <RESULTS_ROOTS[channel]>/<year>/<station>/<channel>/<month_year>/qtransform_<YYYYMMDD>_<pdf_index, 6 digits>.pdf"""
    day = pd.to_datetime(row["date"]).strftime("%Y%m%d")
    return (RESULTS_ROOTS[str(row["channel"])] / str(row["year"]) / str(row["station"]) / str(row["channel"])
            / str(row["month_year"]) / f"qtransform_{day}_{int(row['pdf_index']):06d}.pdf")


# quick check of the path rule, per direction
for ch in sorted(RESULTS_ROOTS):
    _rows = df_emb[df_emb["channel"] == ch]
    _sample = _rows.sample(min(20, len(_rows)), random_state=1)
    _found = sum(qtransform_pdf_path(r).exists() for _, r in _sample.iterrows())
    print(f"{ch}: PDF found for {_found} of {len(_sample)} random rows")

In [ ]:
def render_pdf_first_page(path, width_px=THUMB_PX):
    """Page 1 of a PDF as an RGB array. Uses PyMuPDF, pypdfium2 or pdf2image, whichever is installed."""
    try:
        import fitz                                  # PyMuPDF
    except ImportError:
        fitz = None
    if fitz is not None:
        with fitz.open(path) as doc:
            page = doc[2]
            zoom = width_px / page.rect.width
            pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), colorspace=fitz.csRGB, alpha=False)
            return np.asarray(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
    try:
        import pypdfium2 as pdfium
    except ImportError:
        pdfium = None
    if pdfium is not None:
        pdf = pdfium.PdfDocument(str(path))
        try:
            page = pdf[2]
            return np.asarray(page.render(scale=width_px / page.get_width()).to_pil().convert("RGB"))
        finally:
            pdf.close()
    from pdf2image import convert_from_path
    img = convert_from_path(str(path), dpi=50, first_page=1, last_page=1)[0].convert("RGB")
    return np.asarray(img.resize((width_px, int(img.height * width_px / img.width))))


def pick_thumbnails(E, labels, df_emb, per_class, min_dist_frac, seed):
    """Random points of each class (noise included), spread out so the thumbnails do not pile up.
    Points whose PDF cannot be opened are skipped. Returns a list of (row index, image)."""
    rng = np.random.default_rng(seed)
    En = E / np.ptp(E, axis=0)
    chosen, picks, n_fail = [], [], 0
    for c in np.unique(labels):
        n_taken, fails = 0, 0
        for i in rng.permutation(np.flatnonzero(labels == c)):
            if n_taken >= per_class or fails >= 5 * per_class:
                break
            if chosen and np.hypot(*(En[chosen].T - En[i][:, None])).min() < min_dist_frac:
                continue
            try:
                img = render_pdf_first_page(qtransform_pdf_path(df_emb.iloc[i]))
            except Exception:
                fails += 1
                continue
            chosen.append(i)
            picks.append((i, img))
            n_taken += 1
        n_fail += fails
    if not picks:
        raise RuntimeError("No qtransform PDF could be opened: check RESULTS_ROOT and qtransform_pdf_path().")
    print(f"{len(picks)} thumbnails placed, {n_fail} PDFs skipped (missing or unreadable)")
    return picks

# Balanced Subplot

In [ ]:
import hashlib

# Settings
PER_CLASS = 900          # points drawn from each class
MIN_CLASS_SIZE = 900     # classes with fewer points are dropped (lower it to keep small classes: they contribute all their points)
SAMPLE_SEED = 42
PERPLEXITY_SUB = 100      # None -> sqrt(number of sampled points), the same rule used for the full set

# 1) Draw PER_CLASS points from every class with at least MIN_CLASS_SIZE points (noise, -1, is left out)
rng = np.random.default_rng(SAMPLE_SEED)
class_sizes = pd.Series(labels[labels != -1]).value_counts().sort_index()
kept = class_sizes[class_sizes >= MIN_CLASS_SIZE].index.tolist()
dropped = class_sizes[class_sizes < MIN_CLASS_SIZE].to_dict()
if not kept:
    raise ValueError(f"No class has at least {MIN_CLASS_SIZE} points (class sizes: {class_sizes.to_dict()})")

idx = np.sort(np.concatenate([rng.choice(np.flatnonzero(labels == c), size=min(PER_CLASS, int(class_sizes.loc[c])), replace=False) for c in kept]))

sub_labels = labels[idx]
n_sub = len(idx)
print(f"Classes kept: {kept} | dropped (fewer than {MIN_CLASS_SIZE} points): {dropped}")
print(f"Subsample: {n_sub} points -> {pd.Series(sub_labels).value_counts().sort_index().to_dict()}")

# 2) New t-SNE and UMAP on the features of the sampled points (same preprocessing as the full set), with the cache
X_sub = X_scaled[idx]
perp_sub = PERPLEXITY_SUB or int(round(np.sqrt(n_sub)))
sub_id = hashlib.md5(idx.tobytes()).hexdigest()[:8]      # changes whenever the sampled points change


def run_tsne_sub():
    return TSNE(n_components=2, perplexity=perp_sub, early_exaggeration=EARLY_EXAGGERATION, init="pca",
                learning_rate="auto", random_state=RANDOM_STATE, metric=METRIC).fit_transform(X_sub)

def run_umap_sub():
    return UMAP(n_components=2, n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                metric=METRIC, random_state=RANDOM_STATE).fit_transform(X_sub)


tsne_sub_cache = CACHE_DIR / (f"TERZ_hw12_ont25_sub{n_sub}_{sub_id}_tsne_perp{perp_sub}_ee{EARLY_EXAGGERATION}_{METRIC}_rs{RANDOM_STATE}.npy")
umap_sub_cache = CACHE_DIR / (f"TERZ_hw12_ont25_sub{n_sub}_{sub_id}_umap_nn{UMAP_N_NEIGHBORS}_md{UMAP_MIN_DIST}_{METRIC}_rs{RANDOM_STATE}.npy")
X_tsne_sub = load_or_compute(tsne_sub_cache, run_tsne_sub, n_rows=n_sub, force=FORCE_RECOMPUTE)
X_umap_sub = load_or_compute(umap_sub_cache, run_umap_sub, n_rows=n_sub, force=FORCE_RECOMPUTE)

# 3) Table of the sampled points: metadata + class + the new coordinates (orig_index = row of the full dataset)
df_sub = df_emb.iloc[idx].copy()
df_sub.insert(0, "orig_index", idx)
df_sub["sub_tSNE_1"], df_sub["sub_tSNE_2"] = X_tsne_sub[:, 0], X_tsne_sub[:, 1]
df_sub["sub_umap_1"], df_sub["sub_umap_2"] = X_umap_sub[:, 0], X_umap_sub[:, 1]
df_sub = df_sub.reset_index(drop=True)

print("\nDirections in the subsample (rows = class)")
print(pd.crosstab(df_sub["hdbscan_class"], df_sub["channel"]).to_string())

# 4) Quick look, coloured by HDBSCAN class (same colours as the thumbnail figure)
all_ids = [c for c in np.unique(labels) if c != -1]
cmap = plt.get_cmap("tab10" if len(all_ids) <= 10 else "tab20")
class_color = {c: cmap(k % cmap.N) for k, c in enumerate(all_ids)}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, E_sub, name in zip(axes, (X_tsne_sub, X_umap_sub), ("t-SNE", "UMAP")):
    for c in kept:
        m = sub_labels == c
        ax.scatter(E_sub[m, 0], E_sub[m, 1], s=6, alpha=0.6, color=class_color[c], label=f"class {c}")
    ax.set_xlabel(f"{name} 1")
    ax.set_ylabel(f"{name} 2")
axes[0].set_title(f"t-SNE of the subsample | perplexity = {perp_sub} | {n_sub} points")
axes[1].set_title(f"UMAP of the subsample | n_neighbors = {UMAP_N_NEIGHBORS} | min_dist = {UMAP_MIN_DIST}")
axes[0].legend(markerscale=3, title="HDBSCAN class")
plt.tight_layout()
plt.show()

# Ground velocity of 5 random triggers per class

Same triggers as the seismogram version (same seed and same classes), but the raw counts are converted to ground velocity (m/s) by removing the instrument response, instead of being plotted as counts.

Under every velocity trace there is also the **characteristic function** of the trigger: the recursive STA/LTA of the raw amplitude, with its on-threshold (`CF_THR_ON`) as a dashed line. Set `SHOW_CF = False` to get the figures as before.


In [ ]:
import subprocess
from obspy import read, read_inventory, UTCDateTime
from obspy.clients.fdsn import Client, RoutingClient
from obspy.signal.trigger import recursive_sta_lta

N_PER_CLASS = 5
INCLUDE_NOISE = False        # True: also plot N_PER_CLASS triggers of the HDBSCAN noise (label -1)
WAVEFORM_SEED = 0            # same seed as the counts version -> the same random triggers in every class
PLOT_HALF_WINDOW = 12         # s: the velocity is plotted from trigger - 12 s to trigger + 12 s
DOWNLOAD_HALF_WINDOW = 60    # s: window downloaded around the trigger (the extra time is used to demean/detrend and to remove the response)

MSEED_DIR = None             # folder with the local .mseed files (tried before downloading); None -> only the path stored in df_emb["file"]
MSEED_CACHE_DIR = Path("TERZ_trigger_mseed")        # downloaded ±1 min files (raw counts), the same cache as the counts version
WAVEFORM_FIG_DIR = Path("TERZ_class_velocities")    # one PNG per class
QTRANSFORM_PX = 500                                 # width in pixels of the qtransform image shown next to each velocity trace

FDSNWS_FETCH = ["fdsnws_fetch"]   # or e.g. ["python3", "/path/to/fdsnws_fetch.py"]
FDSN_URL = None                   # None -> fdsnws_fetch default and the EIDA routing service for the response; or e.g. "http://eida.knmi.nl"
NETWORK = "*"                     # put the network code here if the wildcard matches more than one network
LOCATION = "*"

# Instrument response: raw counts -> ground velocity
STATIONXML = None                                   # path (or list of paths) of local StationXML file(s) with the response; None -> download it
STATIONXML_CACHE_DIR = Path("TERZ_stationxml")      # downloaded response, one file per channel
RESPONSE_START, RESPONSE_END = "2022-01-01", "2026-01-01"   # period of the downloaded response (the years of the dataset; every epoch inside it is kept)
PRE_FILT_LOW = (0.05, 0.1)       # Hz: low-frequency cosine taper of the deconvolution
PRE_FILT_HIGH_FRAC = (0.8, 0.9)  # high-frequency cosine taper, as a fraction of the Nyquist frequency (100 Hz data -> 40, 45 Hz)
VEL_BAND = None                  # (fmin, fmax) in Hz to band-pass the velocity before plotting, e.g. (1, 40); None = no extra filter
VEL_SCALE, VEL_UNIT = 1e9, "nm/s"    # remove_response gives m/s; the plots show m/s * VEL_SCALE

# Characteristic function (recursive STA/LTA), drawn under each velocity trace
SHOW_CF = True               # False -> no characteristic function: the figures are exactly as before
CF_STA, CF_LTA = 0.5, 60     # s: short and long windows of the recursive STA/LTA
CF_THR_ON = 25               # on-threshold (the "ont25" of the file names), drawn as a dashed line
CF_WARMUP = 5 * CF_LTA       # s of data before the trigger fed to the STA/LTA, so that the LTA has converged at t = 0 (5 x LTA -> 99 %).
                             # Must be larger than CF_LTA + PLOT_HALF_WINDOW. Triggers with less data than this before them show no CF


In [ ]:
CF_BAND = (2, 30)

# Long-term average (LTA) of the recursive STA/LTA, drawn on the velocity panel (needs SHOW_CF = True: it uses the same CF_WARMUP s of data)
SHOW_LTA = True          # draw +-sqrt(LTA): the RMS amplitude of the background that the STA is compared with, in VEL_UNIT, in the CF_BAND
SHOW_LTA_LEVEL = True    # also draw +-sqrt(CF_THR_ON) x sqrt(LTA): the RMS amplitude that the STA has to reach to cross the on-threshold

from scipy.signal import lfilter


class ResponseError(RuntimeError):
    """The instrument response could not be obtained or applied. It fails the same way for every trigger of the channel, so the loop stops."""


def trigger_utc(row):
    """trigger_time of one row of df_emb as an obspy UTCDateTime."""
    ts = pd.to_datetime(row["trigger_time"], utc=True)
    return UTCDateTime(ts.strftime("%Y-%m-%dT%H:%M:%S.%f"))


def download_mseed(station, channel, t0, t1):
    """Download t0..t1 of one channel with fdsnws_fetch (skipped if it is already in MSEED_CACHE_DIR). Returns the file path."""
    fmt = "%Y-%m-%dT%H:%M:%S"
    start, end = t0.strftime(fmt), (t1 + 1).strftime(fmt)      # whole seconds, so the window is up to 1 s larger than asked
    out = MSEED_CACHE_DIR / f"{station}_{channel}_{start.replace(':', '')}_{end.replace(':', '')}.mseed"
    if out.exists() and out.stat().st_size > 0:
        return out
    MSEED_CACHE_DIR.mkdir(exist_ok=True)
    part = out.with_suffix(".part")                            # write to a temp name first, so a failed download leaves no broken file
    cmd = FDSNWS_FETCH + ["-N", NETWORK, "-S", station, "-L", LOCATION, "-C", channel, "-s", start, "-e", end, "-o", str(part)]
    if FDSN_URL:
        cmd += ["-u", FDSN_URL]
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    except FileNotFoundError:
        raise FileNotFoundError(f"'{FDSNWS_FETCH[0]}' not found: set FDSNWS_FETCH to the right command or script path")
    if res.returncode != 0 or not part.exists() or part.stat().st_size == 0:
        part.unlink(missing_ok=True)
        raise RuntimeError(f"fdsnws_fetch failed (code {res.returncode}): {res.stderr.strip()[-300:]}")
    part.replace(out)
    return out


def read_window(path, station, channel, t0, t1):
    """t0..t1 of one channel from an .mseed file, as a single trace (None if the file has no such data)."""
    st = read(str(path), starttime=t0, endtime=t1).select(station=station, channel=channel)
    if not st:
        return None
    st = st.select(location=sorted({tr.stats.location for tr in st})[0])    # in case several location codes come back
    st.merge(method=1, fill_value="interpolate")
    return st[0]


_inventories = {}


def get_inventory(station, channel):
    """StationXML inventory (with the instrument response) of one channel, for the whole dataset period.
    Order: STATIONXML if it is set, else the cached file in STATIONXML_CACHE_DIR, else a download (FDSN_URL if set,
    otherwise the EIDA routing service) that is then cached."""
    key = (station, channel)
    if key in _inventories:
        return _inventories[key]
    try:
        if STATIONXML:
            files = [STATIONXML] if isinstance(STATIONXML, (str, Path)) else list(STATIONXML)
            inv = read_inventory(str(files[0]))
            for f in files[1:]:
                inv += read_inventory(str(f))
        else:
            out = STATIONXML_CACHE_DIR / f"{station}_{channel}_{RESPONSE_START}_{RESPONSE_END}.xml"
            if out.exists() and out.stat().st_size > 0:
                inv = read_inventory(str(out))
            else:
                client = Client(FDSN_URL) if FDSN_URL else RoutingClient("eida-routing")
                inv = client.get_stations(network=NETWORK, station=station, location=LOCATION, channel=channel,
                                          starttime=UTCDateTime(RESPONSE_START), endtime=UTCDateTime(RESPONSE_END), level="response")
                STATIONXML_CACHE_DIR.mkdir(exist_ok=True)
                part = out.with_suffix(".part")                # write to a temp name first, so a failed download leaves no broken file
                inv.write(str(part), format="STATIONXML")
                part.replace(out)
    except Exception as e:
        raise ResponseError(f"no instrument response for {station} {channel}: {type(e).__name__}: {e}")
    _inventories[key] = inv
    return inv


def get_raw_trace(row):
    """Raw counts of one row of df_emb around the trigger: ±DOWNLOAD_HALF_WINDOW, or from CF_WARMUP s before the trigger if the
    characteristic function is shown and needs more. Reads the local .mseed file if it can, otherwise downloads the window.
    Returns (trace, trigger time)."""
    t = trigger_utc(row)
    pre = max(DOWNLOAD_HALF_WINDOW, CF_WARMUP) if SHOW_CF else DOWNLOAD_HALF_WINDOW
    t0, t1 = t - pre, t + DOWNLOAD_HALF_WINDOW
    station, channel = str(row["station"]), str(row["channel"])

    tr = None
    f = row.get("file")
    if isinstance(f, str) and f:
        for cand in [Path(f)] + ([MSEED_DIR / Path(f).name] if MSEED_DIR else []):
            if cand.is_file():
                try:
                    tr = read_window(cand, station, channel, t0, t1)
                except Exception:
                    tr = None
                if tr is not None:
                    break
    if tr is None:
        tr = read_window(download_mseed(station, channel, t0, t1), station, channel, t0, t1)
    if tr is None:
        raise RuntimeError("no data for this channel and time")

    tr = tr.copy().trim(t0, t1)
    tr.data = tr.data.astype(np.float64)
    return tr, t


def get_velocity(row, raw=None):
    """Ground velocity (m/s) of one row of df_emb. The ±DOWNLOAD_HALF_WINDOW counts are demeaned and detrended, the instrument
    response is removed (remove_response, output="VEL", with a cosine pre-filter), and the trace is cut to ±PLOT_HALF_WINDOW
    around the trigger. Returns (trace, trigger time).
    raw: the (trace, trigger time) already returned by get_raw_trace(row), so the data are read only once (read here if None)."""
    tr, t = raw if raw is not None else get_raw_trace(row)
    tr = tr.copy().trim(t - DOWNLOAD_HALF_WINDOW, t + DOWNLOAD_HALF_WINDOW)   # always the same ±DOWNLOAD_HALF_WINDOW, so the velocity does not depend on the CF
    inv = get_inventory(str(row["station"]), str(row["channel"]))
    tr.detrend("demean")
    tr.detrend("linear")
    nyq = tr.stats.sampling_rate / 2
    pre_filt = (*PRE_FILT_LOW, PRE_FILT_HIGH_FRAC[0] * nyq, PRE_FILT_HIGH_FRAC[1] * nyq)
    try:
        tr.remove_response(inventory=inv, output="VEL", pre_filt=pre_filt, water_level=None,
                           zero_mean=True, taper=True, taper_fraction=0.05)
    except Exception as e:
        raise ResponseError(f"{tr.id} at {tr.stats.starttime}: remove_response failed: {type(e).__name__}: {e}")
    if VEL_BAND:
        tr.filter("bandpass", freqmin=VEL_BAND[0], freqmax=VEL_BAND[1], corners=4, zerophase=True)
    tr.trim(t - PLOT_HALF_WINDOW, t + PLOT_HALF_WINDOW)
    tol = 2 * tr.stats.delta
    if tr.stats.starttime > t - PLOT_HALF_WINDOW + tol or tr.stats.endtime < t + PLOT_HALF_WINDOW - tol:
        raise RuntimeError(f"the data do not cover the whole ±{PLOT_HALF_WINDOW} s window (gap?)")
    return tr, t


_lta_check = {"done": False, "ok": True}


def _sta_lta_parts(data, nsta, nlta):
    """STA and LTA (both in signal units squared) of obspy's recursive_sta_lta: same recursion, x_i = a_i**2,
    sta_i = sta_(i-1) + (x_i - sta_(i-1)) / nsta and the same with nlta for the LTA, started at zero on the second sample.
    recursive_sta_lta only returns their ratio; this gives the LTA itself."""
    x = np.asarray(data, dtype=np.float64) ** 2
    x[0] = 0.0
    sta = lfilter([1.0 / nsta], [1.0, -(1.0 - 1.0 / nsta)], x)
    lta = lfilter([1.0 / nlta], [1.0, -(1.0 - 1.0 / nlta)], x)
    return sta, lta


def get_cf(raw):
    """Characteristic function of one trigger, computed as in paralel_net_proc.py (seismic_trig): the raw counts are band-passed
    with CF_BAND (the detector's FRANGE; obspy bandpass, 4 corners, causal) and then fed to the recursive STA/LTA (obspy) with
    int(CF_STA * sr) and int(CF_LTA * sr) samples. The detector took trigger_time as the first sample where this STA/LTA rises
    above the on-threshold, so the curve should be at CF_THR_ON at t = 0.
    raw: the (trace, trigger time) returned by get_raw_trace(row). Returns the STA/LTA as a trace cut to ±PLOT_HALF_WINDOW around
    the trigger, or None if there are fewer than CF_WARMUP s of data before the trigger (the LTA would not have converged)."""
    tr, t = raw
    if t - tr.stats.starttime < CF_WARMUP - 2 * tr.stats.delta:
        return None
    tr = tr.copy()
    tr.detrend("demean")      # only keeps the start-up transient of the filter small; the detector filters the raw counts directly
    if CF_BAND:
        tr.filter("bandpass", freqmin=CF_BAND[0], freqmax=CF_BAND[1])   # same call as seismic_trig: corners=4, zerophase=False
    sr = tr.stats.sampling_rate
    nsta, nlta = int(CF_STA * sr), int(CF_LTA * sr)
    cft = recursive_sta_lta(tr.data, nsta, nlta)
    if SHOW_LTA and not _lta_check["done"]:                   # once: the LTA that get_lta draws must be the one inside obspy's STA/LTA
        sta, lta = _sta_lta_parts(tr.data, nsta, nlta)
        with np.errstate(divide="ignore", invalid="ignore"):
            ok = bool(np.allclose(sta[nlta:] / lta[nlta:], cft[nlta:], rtol=1e-4, atol=1e-8, equal_nan=True))
        _lta_check.update(done=True, ok=ok)
        if not ok:
            print("WARNING: the STA/LTA recomputed here does not match obspy's recursive_sta_lta, so the LTA is not drawn")
    tr.data = cft
    tr.trim(t - PLOT_HALF_WINDOW, t + PLOT_HALF_WINDOW)
    return tr


def get_lta(row, raw):
    """Long-term average of the recursive STA/LTA of one trigger, as an RMS amplitude: sqrt(LTA) of the ground velocity (response removed
    over the whole CF_WARMUP window, band-passed with CF_BAND like the CF), in VEL_UNIT, cut to ±PLOT_HALF_WINDOW around the trigger.
    The LTA is an exponentially weighted mean of the squared signal (time constant CF_LTA, the sample itself included), so it is not
    the mean of the CF_LTA seconds before the trigger: it also rises while the event is going on.
    raw: the (trace, trigger time) returned by get_raw_trace(row). Returns None if it cannot be computed (same conditions as get_cf)."""
    if not _lta_check["ok"]:
        return None
    tr, t = raw
    if t - tr.stats.starttime < CF_WARMUP - 2 * tr.stats.delta:
        return None
    tr = tr.copy()                                            # the whole window, not only ±DOWNLOAD_HALF_WINDOW: the LTA needs CF_WARMUP s of history
    try:
        inv = get_inventory(str(row["station"]), str(row["channel"]))
        tr.detrend("demean")
        tr.detrend("linear")
        nyq = tr.stats.sampling_rate / 2
        pre_filt = (*PRE_FILT_LOW, PRE_FILT_HIGH_FRAC[0] * nyq, PRE_FILT_HIGH_FRAC[1] * nyq)
        tr.remove_response(inventory=inv, output="VEL", pre_filt=pre_filt, water_level=None,
                           zero_mean=True, taper=True, taper_fraction=0.05)
    except Exception as e:
        print(f"LTA not drawn for {row['channel']} {row['trigger_time']}: {type(e).__name__}: {e}")
        return None
    if CF_BAND:
        tr.filter("bandpass", freqmin=CF_BAND[0], freqmax=CF_BAND[1])      # same call as in get_cf (4 corners, causal)
    sr = tr.stats.sampling_rate
    _, lta = _sta_lta_parts(tr.data, int(CF_STA * sr), int(CF_LTA * sr))
    tr.data = np.sqrt(lta) * VEL_SCALE
    tr.trim(t - PLOT_HALF_WINDOW, t + PLOT_HALF_WINDOW)
    return tr


In [ ]:
cls = df_emb["hdbscan_class"].to_numpy()
class_ids = [c for c in np.unique(cls) if INCLUDE_NOISE or c != -1]
WAVEFORM_FIG_DIR.mkdir(exist_ok=True)
PGV_COL = f"peak_abs_vel_{VEL_UNIT.replace('/', '_')}"          # peak |velocity| of the plotted window, in VEL_UNIT
LTA_COL = f"lta_rms_{VEL_UNIT.replace('/', '_')}"               # sqrt(LTA) at the trigger time, in VEL_UNIT (NaN if not drawn)
lta_band_txt = f"{CF_BAND[0]}-{CF_BAND[1]} Hz" if CF_BAND else "unfiltered"
band_txt = f" | band-pass {VEL_BAND[0]}-{VEL_BAND[1]} Hz" if VEL_BAND else ""

if SHOW_CF and CF_WARMUP < CF_LTA + PLOT_HALF_WINDOW:
    raise ValueError(f"CF_WARMUP ({CF_WARMUP} s) must be larger than CF_LTA + PLOT_HALF_WINDOW ({CF_LTA + PLOT_HALF_WINDOW} s)")

# trigger_time must be an absolute time: its year has to agree with the "year" column
_chk = df_emb.sample(min(50, len(df_emb)), random_state=0)
if (pd.to_datetime(_chk["trigger_time"], utc=True).dt.year.to_numpy() != _chk["year"].astype(int).to_numpy()).mean() > 0.5:
    raise ValueError("trigger_time does not look like an absolute UTC time (its year differs from the 'year' column)")

rng = np.random.default_rng(WAVEFORM_SEED)
sample_rows = []
for c in class_ids:
    name = "noise" if c == -1 else f"class_{c}"
    items, fails = [], 0
    for i in rng.permutation(np.flatnonzero(cls == c)):       # random order; failed rows are replaced by the next one
        if len(items) >= N_PER_CLASS or fails >= 5 * N_PER_CLASS:
            break
        row = df_emb.iloc[i]
        try:
            raw = get_raw_trace(row)                          # read once: used by the velocity and by the characteristic function
            tr, t = get_velocity(row, raw=raw)
            cf = get_cf(raw) if SHOW_CF else None
            lta = get_lta(row, raw) if (SHOW_CF and SHOW_LTA) else None      # after get_cf, which checks the LTA recursion once
        except (FileNotFoundError, ResponseError):
            raise                                             # fdsnws_fetch not found / no usable response: same for every row, no point in trying more
        except Exception as e:
            fails += 1
            print(f"{name}: row {i} skipped ({row['channel']} {row['trigger_time']}): {e}")
            continue
        items.append((i, tr, t, cf, lta))
    if not items:
        raise RuntimeError(f"{name}: no velocity trace could be obtained: check MSEED_DIR, NETWORK, LOCATION and FDSNWS_FETCH")

    n_items = len(items)
    fig = plt.figure(figsize=(15, (3.7 if SHOW_CF else 2.8) * n_items + 0.8), layout="constrained")
    outer = fig.add_gridspec(n_items, 2, width_ratios=[1.7, 1])
    ax_first = None
    for k, (i, tr, t, cf, lta) in enumerate(items):
        if SHOW_CF:                                           # velocity on top, characteristic function right below it (same time axis)
            inner = outer[k, 0].subgridspec(2, 1, height_ratios=[3, 1.3])
            ax = fig.add_subplot(inner[0], sharex=ax_first)
            axc = fig.add_subplot(inner[1], sharex=ax)
        else:
            ax = fig.add_subplot(outer[k, 0], sharex=ax_first)
            axc = None
        axq = fig.add_subplot(outer[k, 1])
        if ax_first is None:
            ax_first = ax
        row = df_emb.iloc[i]
        v = tr.data * VEL_SCALE
        pgv = float(np.abs(v).max())
        ax.plot(tr.times() - (t - tr.stats.starttime), v, color="k", lw=0.7)
        ax.axvline(0, color="tab:red", ls="--", lw=0.8)
        ax.set_xlim(-PLOT_HALF_WINDOW, PLOT_HALF_WINDOW)
        ax.set_ylabel(f"velocity ({VEL_UNIT})")
        lta0 = np.nan
        if lta is not None:                                   # long-term average of the STA/LTA as an RMS amplitude (+-), on the velocity panel
            x_l = lta.times() - (t - lta.stats.starttime)
            lta0 = float(np.interp(0.0, x_l, lta.data))
            ax.plot(x_l, lta.data, color="tab:orange", lw=1.0, label=f"LTA: RMS of the {lta_band_txt} velocity ({CF_LTA:g} s recursive)")
            ax.plot(x_l, -lta.data, color="tab:orange", lw=1.0)
            if SHOW_LTA_LEVEL:
                lvl = np.sqrt(CF_THR_ON) * lta.data
                ax.plot(x_l, lvl, color="tab:green", ls="--", lw=0.9, label=f"sqrt({CF_THR_ON:g}) x LTA: RMS level of the on-threshold")
                ax.plot(x_l, -lvl, color="tab:green", ls="--", lw=0.9)
            if k == 0:
                ax.legend(loc="upper left", fontsize=7, framealpha=0.85)
        lta_txt = f" | LTA = {lta0:.3g} {VEL_UNIT}" if np.isfinite(lta0) else ""
        ax.set_title(f"{row['channel']} | {row['trigger_time']} | row {i} | peak |v| = {pgv:.3g} {VEL_UNIT}{lta_txt}", fontsize=9, loc="left")

        if axc is not None:                                   # characteristic function: recursive STA/LTA of the raw amplitude + on-threshold
            if cf is not None:
                axc.plot(cf.times() - (t - cf.stats.starttime), cf.data, color="tab:blue", lw=0.9,
                         label=f"recursive STA/LTA ({CF_STA:g} s / {CF_LTA:g} s)")
            else:
                axc.text(0.5, 0.5, f"STA/LTA not available (less than {CF_WARMUP:g} s of data before the trigger)",
                         ha="center", va="center", transform=axc.transAxes, fontsize=8)
            axc.axhline(CF_THR_ON, color="tab:red", ls="--", lw=0.9, label=f"on threshold = {CF_THR_ON:g}")
            axc.axvline(0, color="0.4", ls=":", lw=0.8)
            axc.set_ylim(bottom=0)
            axc.set_ylabel("STA/LTA")
            if k == 0:
                axc.legend(loc="upper right", fontsize=7, framealpha=0.85)
            ax.tick_params(labelbottom=False)                 # the time axis is the one of the panel below
        ax_low = axc if axc is not None else ax               # lowest panel of this trigger
        if k < n_items - 1:
            ax_low.tick_params(labelbottom=False)

        axq.axis("off")                                       # qtransform of the same trigger (its PDF from the qtransform folders)
        axq.set_title("qtransform", fontsize=9)
        try:
            axq.imshow(render_pdf_first_page(qtransform_pdf_path(row), width_px=QTRANSFORM_PX))
        except Exception as e:
            axq.text(0.5, 0.5, "qtransform PDF\nnot available", ha="center", va="center", transform=axq.transAxes)
            print(f"{name}: row {i}: qtransform not shown ({type(e).__name__}: {e})")
        sample_rows.append({"hdbscan_class": c, "orig_index": i, "channel": row["channel"], "year": row["year"],
                            "date": row["date"], "trigger_time": row["trigger_time"], "q_value": row["q_value"], "file": row["file"],
                            PGV_COL: pgv, LTA_COL: lta0})
    ax_low.set_xlabel("time from trigger (s)")
    fig.suptitle(f"{'HDBSCAN noise' if c == -1 else f'HDBSCAN class {c}'} | {int((cls == c).sum())} triggers | "
                 f"{len(items)} random examples | ground velocity (instrument response removed){band_txt} | ±{PLOT_HALF_WINDOW} s around the trigger")
    fig.savefig(WAVEFORM_FIG_DIR / f"{name}.png", dpi=150)
    plt.show()
    plt.close(fig)

df_wave = pd.DataFrame(sample_rows)
print(df_wave.to_string(index=False))

# Monthly masks: does a class repeat through the year?

In [ ]:
import calendar
from matplotlib.lines import Line2D

MONTH_BG_MAX = 50000                              # gray background points per panel (random subset, just to keep the figure light)
MONTH_FIG_DIR = Path("TERZ_class_months")         # the two figures are saved here
MONTH_FIG_DIR.mkdir(exist_ok=True)

cls = df_emb["hdbscan_class"].to_numpy()
_d = pd.to_datetime(df_emb["date"])
if _d.isna().any():
    raise ValueError("some rows of df_emb have no valid 'date'")
mon = _d.dt.month.to_numpy()                      # 1..12
yr = _d.dt.year.to_numpy()

# Boolean masks over the rows of df_emb (same order as E and labels)
month_masks = {m: mon == m for m in range(1, 13)}                                          # calendar month, all years together
ym_masks = {(int(y), int(m)): (yr == y) & (mon == m) for y, m in sorted(set(zip(yr, mon)))}  # each month of each year that has data
print(f"{len(month_masks)} calendar-month masks, {len(ym_masks)} year-month masks, "
      f"{sum(int(k.sum()) for k in month_masks.values())} of {len(df_emb)} rows covered")

# Coverage: the number of triggers depends on how much data there is, so rates are triggers per channel-day
# (a channel-day = one direction on one date that has at least one trigger)
chan_days = df_emb.assign(year=yr, month=mon).drop_duplicates(["channel", "date"]).groupby(["year", "month"]).size()
chan_days_month = {m: int(chan_days.xs(m, level="month").sum()) if m in chan_days.index.get_level_values("month") else 0
                   for m in range(1, 13)}

In [ ]:
class_ids = [c for c in np.unique(cls) if c != -1]
_cmap = plt.get_cmap("tab10" if len(class_ids) <= 10 else "tab20")
cls_color = {c: _cmap(k % _cmap.N) for k, c in enumerate(class_ids)}     # same colours as the class figures above
NOISE_COLOR = (0.25, 0.25, 0.25, 1.0)
rgba = np.array([cls_color.get(c, NOISE_COLOR) for c in cls])

bg = np.random.default_rng(0).choice(len(E), size=min(MONTH_BG_MAX, len(E)), replace=False)

fig, axes = plt.subplots(3, 4, figsize=(20, 15), sharex=True, sharey=True)
for m, ax in zip(range(1, 13), axes.ravel()):
    mk = month_masks[m]
    ax.scatter(E[bg, 0], E[bg, 1], s=2, color="lightgray", alpha=0.4, linewidths=0, rasterized=True)
    ax.scatter(E[mk, 0], E[mk, 1], s=4, c=rgba[mk], alpha=0.7, linewidths=0, rasterized=True)
    ax.set_title(f"{calendar.month_name[m]} | {int(mk.sum())} triggers | {chan_days_month[m]} channel-days")
for ax in axes[-1]:
    ax.set_xlabel(f"{emb_name} 1")
for ax in axes[:, 0]:
    ax.set_ylabel(f"{emb_name} 2")
handles = [Line2D([], [], marker="o", ls="", color=cls_color[c], label=f"class {c}") for c in class_ids]
handles.append(Line2D([], [], marker="o", ls="", color=NOISE_COLOR, label="noise"))
fig.legend(handles=handles, loc="center right", title="HDBSCAN class", markerscale=1.5)
fig.suptitle(f"{emb_name} of all triggers, highlighting one calendar month at a time (all years together)", fontsize=16)
fig.tight_layout(rect=[0, 0, 0.93, 0.97])
fig.savefig(MONTH_FIG_DIR / "embedding_by_month.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
counts = pd.crosstab([pd.Series(yr, name="year"), pd.Series(mon, name="month")], pd.Series(cls, name="class"))
rate_ym = counts.div(chan_days.reindex(counts.index), axis=0)                           # triggers per channel-day, per year-month
rate_m = counts.groupby(level="month").sum().div(chan_days.groupby(level="month").sum(), axis=0).reindex(range(1, 13))   # all years together

years = sorted(set(int(y) for y in yr))
year_color = {y: plt.get_cmap("viridis")(i / max(len(years) - 1, 1)) for i, y in enumerate(years)}
ncols = min(4, len(class_ids))
nrows = int(np.ceil(len(class_ids) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.4 * nrows), sharex=True, squeeze=False)
for ax, c in zip(axes.ravel(), class_ids):
    for y in years:
        r = rate_ym.xs(y, level="year")[c].reindex(range(1, 13))
        ax.plot(r.index, r.values, marker="o", ms=3, lw=1, color=year_color[y], label=str(y))
    ax.plot(range(1, 13), rate_m[c].values, color="k", lw=2.2, label="all years")
    ax.set_title(f"class {c} | {int((cls == c).sum())} triggers", color=cls_color[c])
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(list("JFMAMJJASOND"))
    ax.set_ylabel("triggers per channel-day")
    ax.grid(alpha=0.3)
for ax in axes.ravel()[len(class_ids):]:
    ax.axis("off")
axes.ravel()[0].legend(fontsize=8, title="year")
fig.suptitle("Monthly trigger rate of each class (one line per year, black = all years together)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(MONTH_FIG_DIR / "rate_by_month.png", dpi=150)
plt.show()
plt.close(fig)

df_rate_month = rate_m[class_ids].T                      # rows = class, columns = calendar month
df_rate_month.columns = [calendar.month_abbr[m] for m in df_rate_month.columns]
print("Triggers per channel-day, all years together (rows = HDBSCAN class)")
print(df_rate_month.round(2).to_string())

# Thumbnail map: qtransform thumbnails on the embedding

In [ ]:
THUMB_MAP_PATH = Path("TERZ_thumbnail_map.png")

# Thumbnails: N_THUMBS_PER_CLASS random triggers of every class (noise included), spread out so they do not pile up.
# The settings (N_THUMBS_PER_CLASS, MIN_DIST_FRAC, THUMB_PX, THUMB_ZOOM, THUMB_SEED) are the ones defined above.
picks = pick_thumbnails(E, labels, df_emb, N_THUMBS_PER_CLASS, MIN_DIST_FRAC, THUMB_SEED)

all_ids = [c for c in np.unique(labels) if c != -1]
_cmap = plt.get_cmap("tab10" if len(all_ids) <= 10 else "tab20")
class_color = {c: _cmap(k % _cmap.N) for k, c in enumerate(all_ids)}     # same colours as the class figures
NOISE_COLOR = (0.5, 0.5, 0.5, 1.0)

fig, ax = plt.subplots(figsize=(16, 13))
noise = labels == -1
ax.scatter(E[noise, 0], E[noise, 1], s=4, color="lightgray", alpha=0.5, linewidths=0, rasterized=True)
for c in all_ids:
    m = labels == c
    ax.scatter(E[m, 0], E[m, 1], s=4, color=class_color[c], alpha=0.5, linewidths=0, rasterized=True)
for i, img in picks:                                          # thumbnail on its point, framed in the colour of its class
    ab = AnnotationBbox(OffsetImage(img, zoom=THUMB_ZOOM), (E[i, 0], E[i, 1]), frameon=True, pad=0.05,
                        bboxprops=dict(edgecolor=class_color.get(labels[i], NOISE_COLOR), linewidth=1.8))
    ax.add_artist(ab)

handles = [Line2D([], [], marker="o", ls="", color=class_color[c], label=f"class {c} ({int((labels == c).sum())})") for c in all_ids]
handles.append(Line2D([], [], marker="o", ls="", color=NOISE_COLOR, label=f"noise ({int(noise.sum())})"))
ax.legend(handles=handles, loc="best", title="HDBSCAN class", markerscale=1.5)
ax.set_xlabel(f"{emb_name} 1")
ax.set_ylabel(f"{emb_name} 2")
ax.set_title(f"{emb_name} | {len(picks)} qtransform thumbnails ({N_THUMBS_PER_CLASS} random per class, frame = class colour) | {len(E)} triggers")
fig.tight_layout()
fig.savefig(THUMB_MAP_PATH, dpi=150)
plt.show()
plt.close(fig)